# Experimento da máscara de plantas

Notebook para testar visualmente a máscara automática usada no relatório de iluminação.

A máscara é heurística: prioriza pixels verdes e saturados e tenta excluir terra, vasos e piso. Ajuste os limiares abaixo usando fotos reais da estufa.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PHOTO_DIR = Path('../api/data/captures/interval')
photos = sorted(PHOTO_DIR.glob('interval_*.jp*g'))
print(f'{len(photos)} fotos encontradas em {PHOTO_DIR.resolve()}')
if not photos:
    raise FileNotFoundError('Baixe ou copie fotos interval_YYYYMMDD_HHMMSS.jpg para api/data/captures/interval')

In [ ]:
# Limiar inicial, igual ao usado no backend. Ajuste e execute novamente.
EXCESS_GREEN_MIN = 18
GREEN_RATIO_MIN = 0.90
SATURATION_MIN = 0.12
BRIGHTNESS_MIN = 35

def load_rgb(path, max_width=1200):
    with Image.open(path) as source:
        image = source.convert('RGB')
    if image.width > max_width:
        image.thumbnail((max_width, max_width), Image.Resampling.LANCZOS)
    return np.asarray(image)

def vegetation_mask(image):
    channels = image.astype(np.float32)
    red, green, blue = channels[..., 0], channels[..., 1], channels[..., 2]
    maximum = channels.max(axis=2)
    minimum = channels.min(axis=2)
    saturation = (maximum - minimum) / np.maximum(maximum, 1)
    excess_green = 2 * green - red - blue
    return (
        (excess_green >= EXCESS_GREEN_MIN)
        & (green >= red * GREEN_RATIO_MIN)
        & (green >= blue * GREEN_RATIO_MIN)
        & (saturation >= SATURATION_MIN)
        & (maximum >= BRIGHTNESS_MIN)
    )

def show_mask(path):
    image = load_rgb(path)
    mask = vegetation_mask(image)
    overlay = image.copy()
    overlay[~mask] = (overlay[~mask] * 0.18).astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image); axes[0].set_title(path.name); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray', vmin=0, vmax=1); axes[1].set_title(f'Máscara: {mask.mean():.1%}'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Somente plantas em destaque'); axes[2].axis('off')
    plt.tight_layout()
    return mask

mask = show_mask(photos[len(photos) // 2])

In [ ]:
# Compare algumas fotos ao longo do dia.
sample = photos[::max(1, len(photos) // 6)][:6]
fig, axes = plt.subplots(len(sample), 3, figsize=(15, 4 * len(sample)))
if len(sample) == 1:
    axes = np.asarray([axes])
for row, path in enumerate(sample):
    image = load_rgb(path)
    mask = vegetation_mask(image)
    overlay = image.copy()
    overlay[~mask] = (overlay[~mask] * 0.18).astype(np.uint8)
    axes[row, 0].imshow(image); axes[row, 0].set_title(path.name); axes[row, 0].axis('off')
    axes[row, 1].imshow(mask, cmap='gray', vmin=0, vmax=1); axes[row, 1].set_title(f'{mask.mean():.1%} classificado'); axes[row, 1].axis('off')
    axes[row, 2].imshow(overlay); axes[row, 2].set_title('Overlay'); axes[row, 2].axis('off')
plt.tight_layout()

In [ ]:
# Diagnóstico: cobertura da máscara em todas as fotos.
coverages = []
for path in photos:
    coverages.append(vegetation_mask(load_rgb(path)).mean())

plt.figure(figsize=(12, 4))
plt.plot(coverages, marker='o', ms=3)
plt.ylim(0, 1)
plt.ylabel('fração classificada como planta')
plt.xlabel('foto em ordem cronológica')
plt.grid(alpha=0.25)
plt.show()
print(f'mínimo={min(coverages):.1%}, mediana={np.median(coverages):.1%}, máximo={max(coverages):.1%}')

## Como ajustar

- Incluiu terra ou vasos? Aumente `EXCESS_GREEN_MIN` ou `SATURATION_MIN`.
- Perdeu folhas escuras? Reduza `BRIGHTNESS_MIN` ou `EXCESS_GREEN_MIN`.
- Perdeu folhas amareladas? Reduza `GREEN_RATIO_MIN` para algo como `0.80`.
- A máscara ficou quase vazia? Verifique primeiro se a imagem tem plantas verdes visíveis e se a iluminação não está muito colorida.

A máscara do backend usa os mesmos quatro limiares principais. Depois de encontrar valores melhores, copie-os para `_vegetation_mask` em `api/server.py` e incremente `ILLUMINATION_ANALYSIS_VERSION` para forçar uma nova análise.